<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-10/nlp-pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

print("Setup done.")

Setup done.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
class PreprocessingModule:
    """
    Handles cleaning and normalizing raw text before vectorization.

    Parameters:
        remove_stopwords (bool): whether to strip common low-signal words. Default True.
        lemmatize (bool): whether to reduce words to their dictionary base form. Default True.
    """
    def __init__(self, remove_stopwords=True, lemmatize=True):
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        """Takes a raw string, returns a cleaned string ready for vectorization."""
        if not isinstance(text, str):
            raise TypeError("Input must be a string.")

        # lowercase + strip punctuation
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)

        tokens = word_tokenize(text)

        if self.remove_stopwords:
            tokens = [t for t in tokens if t not in self.stop_words]

        if self.lemmatize:
            tokens = [self.lemmatizer.lemmatize(t) for t in tokens]

        return " ".join(tokens)

In [5]:
class VectorizerModule:
    """
    Handles TF-IDF vectorization and similarity scoring.
    """
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None

    def fit(self, corpus):
        """Fits the vectorizer on a list of (already preprocessed) documents."""
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)
        return self

    def transform(self, query):
        """Transforms a single (preprocessed) query string into a vector."""
        return self.vectorizer.transform([query])

    def similarity(self, query_vector):
        """Returns cosine similarity scores between the query and the fitted corpus."""
        return cosine_similarity(query_vector, self.corpus_vectors)[0]

In [6]:
class Pipeline:
    """
    Chains PreprocessingModule and VectorizerModule into a single reusable
    text-similarity search pipeline.
    """
    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer_module = VectorizerModule()
        self.raw_corpus = None

    def fit(self, corpus):
        """Preprocesses and fits the vectorizer on the corpus."""
        self.raw_corpus = corpus
        cleaned_corpus = [self.preprocessor.transform(doc) for doc in corpus]
        self.vectorizer_module.fit(cleaned_corpus)
        return self

    def run(self, query, corpus, top_k=3):
        """
        Runs the full pipeline: preprocess query -> vectorize -> rank by similarity.
        Returns a list of (score, original_document) tuples, highest first.
        """
        # --- edge case handling ---
        if not isinstance(query, str) or query.strip() == "":
            raise ValueError("Query cannot be empty.")
        if len(query.strip()) == 1:
            raise ValueError("Query is too short to be meaningful (single character).")
        if re.fullmatch(r'[\d\W_]+', query.strip()):
            raise ValueError("Query must contain at least one alphabetic word.")

        if self.raw_corpus != corpus:
            self.fit(corpus)

        cleaned_query = self.preprocessor.transform(query)
        query_vector = self.vectorizer_module.transform(cleaned_query)
        scores = self.vectorizer_module.similarity(query_vector)

        ranked = sorted(zip(scores, corpus), reverse=True, key=lambda x: x[0])
        return ranked[:top_k]

In [7]:
corpus = [
    "The dog ran happily across the park.",
    "My puppy loves to play fetch every morning.",
    "Cats and dogs are the most common household pets.",
    "The kitten curled up and slept all afternoon.",
    "The new smartphone has an impressive camera.",
    "Artificial intelligence is transforming the tech industry.",
    "Laptops with faster processors improve productivity.",
    "The software update fixed several major bugs.",
    "The airplane landed safely despite the storm.",
    "We booked flights for our vacation to Europe.",
    "The airport was crowded during the holiday season.",
    "Traveling by train offers scenic views of the countryside.",
    "The chef prepared a delicious pasta dish.",
    "She baked chocolate chip cookies for the party.",
    "The restaurant received excellent reviews for its service."
]

pipeline = Pipeline()
pipeline.fit(corpus)

test_queries = [
    "I have a new puppy at home",
    "My laptop's battery drains too fast",
    "Booking a flight for the holidays",
    "This restaurant serves amazing food",
    "A cat sleeping on the couch"
]

for query in test_queries:
    print(f"Query: '{query}'")
    results = pipeline.run(query, corpus, top_k=3)
    for score, doc in results:
        print(f"  [{score:.4f}] {doc}")
    print()

Query: 'I have a new puppy at home'
  [0.3536] The new smartphone has an impressive camera.
  [0.2887] My puppy loves to play fetch every morning.
  [0.0000] The dog ran happily across the park.

Query: 'My laptop's battery drains too fast'
  [0.4472] Laptops with faster processors improve productivity.
  [0.0000] The dog ran happily across the park.
  [0.0000] My puppy loves to play fetch every morning.

Query: 'Booking a flight for the holidays'
  [0.3536] We booked flights for our vacation to Europe.
  [0.3536] The airport was crowded during the holiday season.
  [0.0000] The dog ran happily across the park.

Query: 'This restaurant serves amazing food'
  [0.4472] The restaurant received excellent reviews for its service.
  [0.0000] The dog ran happily across the park.
  [0.0000] My puppy loves to play fetch every morning.

Query: 'A cat sleeping on the couch'
  [0.4586] Cats and dogs are the most common household pets.
  [0.0000] The dog ran happily across the park.
  [0.0000] My p

In [8]:
edge_cases = [
    ("", "empty string"),
    ("x", "single-character query"),
    ("12345 !!!", "numbers/symbols only")
]

for query, description in edge_cases:
    print(f"Testing: {description} -> '{query}'")
    try:
        pipeline.run(query, corpus, top_k=3)
        print("  No error raised (unexpected)")
    except ValueError as e:
        print(f"  Correctly raised ValueError: {e}")
    print()

Testing: empty string -> ''
  Correctly raised ValueError: Query cannot be empty.

Testing: single-character query -> 'x'
  Correctly raised ValueError: Query is too short to be meaningful (single character).

Testing: numbers/symbols only -> '12345 !!!'
  Correctly raised ValueError: Query must contain at least one alphabetic word.

